# Curated VG150 Pipeline (Feature Extraction in RAM)
This notebook parses `VG-SGG.h5`, crops bounding boxes in RAM, extracts ViT, RoBERTa, and ZipDepth features, and outputs `ent_rgb_emb.pt`, `ent_depth_emb.pt`, and `ent_text_emb.pt`.

In [ ]:
!pip install h5py transformers timm opencv-python
!cp -r /kaggle/input/kaggle-payload/core /kaggle/working/ 2>/dev/null || true
!cp -r /kaggle/input/kaggle-payload/ZipDepth /kaggle/working/ZipDepth 2>/dev/null || true
!cp -r /kaggle/input/kaggle-payload/zipdepth /kaggle/working/ZipDepth 2>/dev/null || true
!pip install /kaggle/working/ZipDepth


In [ ]:
%%writefile process_vg_kaggle.py
import os
import sys
import json
import h5py
import torch
import numpy as np
from PIL import Image
import torchvision.transforms as T
from transformers import RobertaTokenizer
from tqdm import tqdm

# Ensure packages are visible
sys.path.append('/kaggle/working')

# --- Models ---
from core.features.encoders import TextEncoder, RGBEncoder, DepthEncoder
from zipdepth.model.architecture import create_model

def main():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")
    
    # --- Auto-detect VG_IMAGES_DIR using os.walk (100% foolproof) ---
    VG_IMAGES_DIR = None
    VG_IMAGES2_DIR = None
    IMAGE_DATA_JSON_PATH = None
    
    for root, dirs, files in os.walk('/kaggle/input'):
        if 'VG_100K' in dirs:
            VG_IMAGES_DIR = os.path.join(root, 'VG_100K')
        if 'VG_100K_2' in dirs:
            VG_IMAGES2_DIR = os.path.join(root, 'VG_100K_2')
        if 'image_data.json' in files and IMAGE_DATA_JSON_PATH is None:
            IMAGE_DATA_JSON_PATH = os.path.join(root, 'image_data.json')
            
    if VG_IMAGES_DIR and VG_IMAGES2_DIR:
        print(f"Auto-detected VG_IMAGES_DIR: {VG_IMAGES_DIR}")
        print(f"Auto-detected VG_IMAGES2_DIR: {VG_IMAGES2_DIR}")
    else:
        print("CRITICAL ERROR: Could not find VG_100K directory in Kaggle inputs!")
        sys.exit(1)
        
    if IMAGE_DATA_JSON_PATH:
        print(f"Auto-detected image_data.json: {IMAGE_DATA_JSON_PATH}")
    else:
        print("CRITICAL ERROR: Could not find image_data.json in Kaggle inputs!")
        sys.exit(1)
        
    VG_H5_PATH = '/kaggle/input/kaggle-payload/VG-SGG.h5'
    VG_DICT_PATH = '/kaggle/input/kaggle-payload/VG-SGG-dicts.json'
    
    # Handle Kaggle's lowercasing of dataset paths
    ZIPDEPTH_CKPT = '/kaggle/working/ZipDepth/checkpoints/zipdepth_base.pth'
    if not os.path.exists(ZIPDEPTH_CKPT):
        ZIPDEPTH_CKPT = '/kaggle/input/kaggle-payload/zipdepth/checkpoints/zipdepth_base.pth'
    
    OUT_DIR = '/kaggle/working/Spatial-VG'
    os.makedirs(OUT_DIR, exist_ok=True)
    
    # --- Load Image Metadata ---
    print("Loading image_data.json to map filenames...")
    with open(IMAGE_DATA_JSON_PATH, 'r') as f:
        im_data = json.load(f)
        
    valid_images = []
    corrupted_ims = {'1592.jpg', '1722.jpg', '4616.jpg', '4617.jpg'}
    for img in im_data:
        basename = str(img['image_id']) + '.jpg'
        if basename not in corrupted_ims:
            valid_images.append(basename)
            
    # --- Load Models ---
    print("Loading Encoders and ZipDepth...")
    text_enc = TextEncoder(out_dim=768).to(device).eval()
    rgb_enc = RGBEncoder(out_dim=768, pretrained=True).to(device).eval()
    depth_enc = DepthEncoder(out_dim=768).to(device).eval()
    tokenizer = RobertaTokenizer.from_pretrained('roberta-base')
    
    # Load ZipDepth
    zipdepth = create_model('zipdepth_base', pretrained=False)
    checkpoint = torch.load(ZIPDEPTH_CKPT, map_location='cpu')
    sd = checkpoint.get('model_state_dict', checkpoint)
    clean_sd = {k.replace('module.', ''): v for k, v in sd.items()}
    zipdepth.load_state_dict(clean_sd, strict=False)
    zipdepth = zipdepth.to(device).eval()
    
    zipdepth_transform = T.Compose([
        T.Resize((384, 384)),
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    rgb_transform = T.Compose([
        T.Resize((224, 224)),
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    depth_transform = T.Compose([
        T.Resize((224, 224)),
        T.ToTensor(),
    ])
    
    # --- Load Dictionaries ---
    with open(VG_DICT_PATH, 'r') as f:
        info = json.load(f)
    idx_to_label = info['idx_to_label']
    idx_to_predicate = info['idx_to_predicate']
    
    # --- Parse H5 ---
    print("Parsing VG-SGG.h5...")
    h5 = h5py.File(VG_H5_PATH, 'r')
    boxes_1024 = h5['boxes_1024'][:]
    labels = h5['labels'][:]
    relationships = h5['relationships'][:]
    predicates = h5['predicates'][:]
    
    img_to_first_box = h5['img_to_first_box'][:]
    img_to_last_box = h5['img_to_last_box'][:]
    img_to_first_rel = h5['img_to_first_rel'][:]
    img_to_last_rel = h5['img_to_last_rel'][:]
    split = h5['split'][:]
    
    num_images = len(split)
    num_boxes = len(boxes_1024)
    
    print(f"Total Images: {num_images} | Total Boxes: {num_boxes}")
    assert len(valid_images) == num_images, "Mismatch between image_data.json and H5!"
    
    all_rgb_emb = torch.zeros((num_boxes, 768), dtype=torch.float32)
    all_depth_emb = torch.zeros((num_boxes, 768), dtype=torch.float32)
    all_text_emb = torch.zeros((num_boxes, 768), dtype=torch.float32)
    
    train_triplets = []
    test_triplets = []
    
    warning_printed = False
    with torch.no_grad():
        for i in tqdm(range(num_images), desc="Extracting in RAM"):
            first_box = img_to_first_box[i]
            last_box = img_to_last_box[i]
            if first_box < 0 or last_box < 0:
                continue
                
            img_name = valid_images[i]
            img_path = os.path.join(VG_IMAGES_DIR, img_name)
            if not os.path.exists(img_path):
                img_path = os.path.join(VG_IMAGES2_DIR, img_name)
            
            if not os.path.exists(img_path):
                if not warning_printed:
                    print(f"\nCRITICAL WARNING: Image {img_name} not found!")
                    print(f"Tried looking in {VG_IMAGES_DIR} and {VG_IMAGES2_DIR}")
                    print(f"Are you sure the Dataset is properly attached?")
                    warning_printed = True
                continue
                
            img_pil = Image.open(img_path).convert('RGB')
            w, h = img_pil.size
            
            # Global Depth via ZipDepth
            input_batch = zipdepth_transform(img_pil).unsqueeze(0).to(device)
            prediction = zipdepth(input_batch)
            prediction = torch.nn.functional.interpolate(
                prediction,
                size=(h, w),
                mode="bicubic",
                align_corners=False,
            ).squeeze()
            depth_map = prediction.cpu().numpy()
            d_min, d_max = depth_map.min(), depth_map.max()
            if d_max > d_min: depth_map = (depth_map - d_min) / (d_max - d_min)
            depth_img = Image.fromarray((depth_map * 255).astype(np.uint8), mode='L')
            
            for b_idx in range(first_box, last_box + 1):
                box = boxes_1024[b_idx]
                xc, yc, bw, bh = box
                xmin = max(0, int((xc - bw/2) / 1024 * w))
                ymin = max(0, int((yc - bh/2) / 1024 * h))
                xmax = min(w, int((xc + bw/2) / 1024 * w))
                ymax = min(h, int((yc + bh/2) / 1024 * h))
                
                c_rgb = img_pil.crop((xmin, ymin, xmax, ymax))
                c_depth = depth_img.crop((xmin, ymin, xmax, ymax))
                
                rgb_t = rgb_transform(c_rgb).unsqueeze(0).to(device)
                rgb_feat = rgb_enc(rgb_t).squeeze(0).cpu()
                all_rgb_emb[b_idx] = rgb_feat
                
                depth_t = depth_transform(c_depth).unsqueeze(0).to(device)
                depth_feat = depth_enc(depth_t).squeeze(0).cpu()
                all_depth_emb[b_idx] = depth_feat
                
                label_id = str(labels[b_idx][0])
                label_text = idx_to_label.get(label_id, 'unknown')
                tokens = tokenizer(label_text, padding='max_length', truncation=True, max_length=16, return_tensors='pt')
                text_feat = text_enc(tokens['input_ids'].to(device), tokens['attention_mask'].to(device)).squeeze(0).cpu()
                all_text_emb[b_idx] = text_feat
            
            first_rel = img_to_first_rel[i]
            last_rel = img_to_last_rel[i]
            if first_rel >= 0 and last_rel >= 0:
                for r_idx in range(first_rel, last_rel + 1):
                    subj, obj = relationships[r_idx]
                    pred_id = str(predicates[r_idx][0])
                    
                    trip = f"{subj}\t{obj}\t{pred_id}\n"
                    if split[i] == 0:
                        train_triplets.append(trip)
                    elif split[i] == 2:
                        test_triplets.append(trip)
    
    print("Saving Tensors to Disk...")
    torch.save(all_rgb_emb, os.path.join(OUT_DIR, 'ent_rgb_emb.pt'))
    torch.save(all_depth_emb, os.path.join(OUT_DIR, 'ent_depth_emb.pt'))
    torch.save(all_text_emb, os.path.join(OUT_DIR, 'ent_text_emb.pt'))
    
    with open(os.path.join(OUT_DIR, 'train.txt'), 'w') as f:
        f.write(f"{len(train_triplets)}\n")
        f.writelines(train_triplets)
    with open(os.path.join(OUT_DIR, 'test.txt'), 'w') as f:
        f.write(f"{len(test_triplets)}\n")
        f.writelines(test_triplets)
        
    print("All Done!")

if __name__ == '__main__':
    main()


In [ ]:
!python process_vg_kaggle.py